In [65]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.model_selection import cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import KFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OrdinalEncoder


In [66]:
dataset = pd.read_csv('/content/Housing.csv')

In [67]:
dataset

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished
...,...,...,...,...,...,...,...,...,...,...,...,...,...
540,1820000,3000,2,1,1,yes,no,yes,no,no,2,no,unfurnished
541,1767150,2400,3,1,1,no,no,no,no,no,0,no,semi-furnished
542,1750000,3620,2,1,1,yes,no,no,no,no,0,no,unfurnished
543,1750000,2910,3,1,1,no,no,no,no,no,0,no,furnished


In [68]:
dataset.describe()

,price,area,bedrooms,bathrooms,stories,parking
count,5.450000e+02,545.000000,545.000000,545.000000,545.000000,545.000000
mean,4.766729e+06,5150.541284,2.965138,1.286239,1.805505,0.693578
std,1.870440e+06,2170.141023,0.738064,0.502470,0.867492,0.861586
min,1.750000e+06,1650.000000,1.000000,1.000000,1.000000,0.000000
25%,3.430000e+06,3600.000000,2.000000,1.000000,1.000000,0.000000
50%,4.340000e+06,4600.000000,3.000000,1.000000,2.000000,0.000000
75%,5.740000e+06,6360.000000,3.000000,2.000000,2.000000,1.000000
max,1.330000e+07,16200.000000,6.000000,4.000000,4.000000,3.000000


In [69]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 545 entries, 0 to 544
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   price             545 non-null    int64 
 1   area              545 non-null    int64 
 2   bedrooms          545 non-null    int64 
 3   bathrooms         545 non-null    int64 
 4   stories           545 non-null    int64 
 5   mainroad          545 non-null    object
 6   guestroom         545 non-null    object
 7   basement          545 non-null    object
 8   hotwaterheating   545 non-null    object
 9   airconditioning   545 non-null    object
 10  parking           545 non-null    int64 
 11  prefarea          545 non-null    object
 12  furnishingstatus  545 non-null    object
dtypes: int64(6), object(7)
memory usage: 55.5+ KB


In [70]:
dataset.columns

Index(['price', 'area', 'bedrooms', 'bathrooms', 'stories', 'mainroad',
       'guestroom', 'basement', 'hotwaterheating', 'airconditioning',
       'parking', 'prefarea', 'furnishingstatus'],
      dtype='object')

In [71]:
dataset.furnishingstatus.unique()

array(['furnished', 'semi-furnished', 'unfurnished'], dtype=object)

In [72]:
one_hot_encode_cols = ['mainroad','guestroom','basement','hotwaterheating','airconditioning','prefarea']
ordinal_encode_cols = ['furnishingstatus']
furnish_order = ['unfurnished','semi-furnished','furnished']

In [73]:
ct = ColumnTransformer(transformers=[
    ('one_hot', OneHotEncoder(drop = 'if_binary'), one_hot_encode_cols),
    ('ordinal_encode', OrdinalEncoder(categories = [furnish_order]), ordinal_encode_cols)
], remainder='passthrough')

In [74]:
transformed_dataset = ct.fit_transform(dataset)

In [75]:
transformed_dataset

array([[1., 0., 0., ..., 2., 3., 2.],
       [1., 0., 0., ..., 4., 4., 3.],
       [1., 0., 1., ..., 2., 2., 2.],
       ...,
       [1., 0., 0., ..., 1., 1., 0.],
       [0., 0., 0., ..., 1., 1., 0.],
       [1., 0., 0., ..., 1., 2., 0.]])

In [76]:
raw_col_names = ct.get_feature_names_out()
col_names = [name.split('__')[-1] for name in raw_col_names]
col_names

['mainroad_yes',
 'guestroom_yes',
 'basement_yes',
 'hotwaterheating_yes',
 'airconditioning_yes',
 'prefarea_yes',
 'furnishingstatus',
 'price',
 'area',
 'bedrooms',
 'bathrooms',
 'stories',
 'parking']

In [77]:
new_dataset = pd.DataFrame(transformed_dataset, columns = col_names)

In [78]:
new_dataset

,mainroad_yes,guestroom_yes,basement_yes,hotwaterheating_yes,airconditioning_yes,prefarea_yes,furnishingstatus,price,area,bedrooms,bathrooms,stories,parking
0,1.0,0.0,0.0,0.0,1.0,1.0,2.0,13300000.0,7420.0,4.0,2.0,3.0,2.0
1,1.0,0.0,0.0,0.0,1.0,0.0,2.0,12250000.0,8960.0,4.0,4.0,4.0,3.0
2,1.0,0.0,1.0,0.0,0.0,1.0,1.0,12250000.0,9960.0,3.0,2.0,2.0,2.0
3,1.0,0.0,1.0,0.0,1.0,1.0,2.0,12215000.0,7500.0,4.0,2.0,2.0,3.0
4,1.0,1.0,1.0,0.0,1.0,0.0,2.0,11410000.0,7420.0,4.0,1.0,2.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
540,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1820000.0,3000.0,2.0,1.0,1.0,2.0
541,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1767150.0,2400.0,3.0,1.0,1.0,0.0
542,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1750000.0,3620.0,2.0,1.0,1.0,0.0
543,0.0,0.0,0.0,0.0,0.0,0.0,2.0,1750000.0,2910.0,3.0,1.0,1.0,0.0


In [79]:
x, y = new_dataset.drop(['price'], axis = 1), new_dataset.price

In [80]:
x

,mainroad_yes,guestroom_yes,basement_yes,hotwaterheating_yes,airconditioning_yes,prefarea_yes,furnishingstatus,area,bedrooms,bathrooms,stories,parking
0,1.0,0.0,0.0,0.0,1.0,1.0,2.0,7420.0,4.0,2.0,3.0,2.0
1,1.0,0.0,0.0,0.0,1.0,0.0,2.0,8960.0,4.0,4.0,4.0,3.0
2,1.0,0.0,1.0,0.0,0.0,1.0,1.0,9960.0,3.0,2.0,2.0,2.0
3,1.0,0.0,1.0,0.0,1.0,1.0,2.0,7500.0,4.0,2.0,2.0,3.0
4,1.0,1.0,1.0,0.0,1.0,0.0,2.0,7420.0,4.0,1.0,2.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...
540,1.0,0.0,1.0,0.0,0.0,0.0,0.0,3000.0,2.0,1.0,1.0,2.0
541,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2400.0,3.0,1.0,1.0,0.0
542,1.0,0.0,0.0,0.0,0.0,0.0,0.0,3620.0,2.0,1.0,1.0,0.0
543,0.0,0.0,0.0,0.0,0.0,0.0,2.0,2910.0,3.0,1.0,1.0,0.0


In [81]:
y

,price
0,13300000.0
1,12250000.0
2,12250000.0
3,12215000.0
4,11410000.0
...,...
540,1820000.0
541,1767150.0
542,1750000.0
543,1750000.0


In [82]:
models = [
    ('LinearReg', LinearRegression()),
    ('RandomForest', RandomForestRegressor()),
    ('DecisionTree', DecisionTreeRegressor()),
    ('xgbregressor', XGBRegressor()),
    ('GradientBoost', GradientBoostingRegressor())
]

In [83]:
results = []
kf = KFold(n_splits=5, shuffle=True, random_state = 43)

for name, model in models:
  training_model = Pipeline(steps=[
      ('scaler', StandardScaler()),
      (name, model)
  ])

  scores = cross_val_score(training_model, x, y, cv = kf, scoring = 'r2')

  results.append({
      'Model_Name' : name,
      'Mean_Squared_error_percentage' : scores.mean(),
      'std dev' : scores.std()
  })
result_df = pd.DataFrame(results)

In [84]:
result_df

,Model_Name,Mean_Squared_error_percentage,std dev
0,LinearReg,0.657797,0.047722
1,RandomForest,0.618308,0.029529
2,DecisionTree,0.280271,0.130440
3,xgbregressor,0.559029,0.032108
4,GradientBoost,0.651258,0.037355


In [85]:
xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size = 0.2, random_state = 43)

In [86]:
model = LinearRegression()
model.fit(xtrain, ytrain)

LinearRegression()

In [87]:
ypreds = model.predict(xtest)
accuracy = r2_score(ypreds, ytest)

In [88]:
print('Accuracy of the model is : ', accuracy)

Accuracy of the model is :  0.35798452309874995
